# Ragas Hallucination Evaluation

The purpose of this notebook is to run test cases of hallucination evaluations on datasets.

## Setup

In [10]:
import pandas as pd
import json
import copy
import numpy as np
from dotenv import load_dotenv
import os
import random
from datasets import Dataset
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, faithfulness
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from langchain_anthropic import ChatAnthropic
import asyncio
import time
from tqdm.asyncio import tqdm_asyncio
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## LLM Link

In [2]:
# Load environment files and store API keys
load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_KEY")

# Define Claude model
claude_model = "claude-3-5-haiku-20241022"

## Data

In [3]:
file_path = "../../data/Hallucination/qa_data.json"
with open(file_path, "r", encoding="utf-8") as file:
    data = [json.loads(line) for line in file]

In [4]:
random.seed(42)

records = []
hallucination_flags = []
answers = []

# Sample 1000 random entries from data
sampled_entries = random.sample(data, 1000)

for entry in sampled_entries:
    is_hallucinated = random.choice([True, False])
    actual_output = entry["hallucinated_answer"] if is_hallucinated else entry["right_answer"]    
    records.append({
        "input": entry["question"],
        "actual_output": actual_output,
        "context": entry["knowledge"]
    })
    hallucination_flags.append(1 if is_hallucinated else 0)
    answers.append(actual_output)

# Convert to DataFrame
df = pd.DataFrame(records).reset_index()

hallucination_df = pd.DataFrame({
    "index": df["index"],
    "hallucinated": hallucination_flags,
    "actual_output": answers
})

In [5]:
hallucination_df.hallucinated.value_counts()

hallucinated
0    534
1    466
Name: count, dtype: int64

## Ragas Evaluation

In [6]:
# Set up Claude as the evaluator LLM
claude_llm = ChatAnthropic(api_key=anthropic_key, model=claude_model)
evaluator_llm = LangchainLLMWrapper(claude_llm)

In [15]:
# Function to run evaluations
async def evaluate_hallucination(index, context, question, response):
    from ragas.dataset_schema import SingleTurnSample
    from ragas.metrics import Faithfulness
    
    try:
        sample = SingleTurnSample(
            user_input=question,
            response=response,
            retrieved_contexts=context
        )
        
        scorer = Faithfulness(llm=evaluator_llm)
        faithfulness_score = await scorer.single_turn_ascore(sample)
        return (index, faithfulness_score)
        
    except Exception as e:
        error_str = str(e)
        # Check if this is a rate limit error (error code 429)
        if "429" in error_str or "rate_limit" in error_str.lower():
            wait_time = 1
            print(f"Rate limit hit, waiting for {wait_time} seconds before retry")
            await asyncio.sleep(wait_time)
        else:
            # If it's a different error, just raise it
            raise e

In [23]:
SEMAPHORE_LIMIT = 50
semaphore = asyncio.Semaphore(SEMAPHORE_LIMIT)

# Wrap with semaphore to limit concurrency
async def evaluate_with_limit(index, context, question, response):
    async with semaphore:
        return await evaluate_hallucination(index, context, question, response)

# Run all evaluations in a controlled, batched way
async def evaluate_all(df):
    tasks = []
    for index, row in df.iterrows():
        index = row['index']
        context = [row.context]
        question = row.input
        response = row.actual_output
        tasks.append(evaluate_with_limit(index, context, question, response))

    results = []
    for coro in tqdm_asyncio.as_completed(tasks, total=len(tasks), desc="Evaluating"):
        try:
            result = await coro
            results.append(result)
        except Exception as e:
            results.append(f"Error: {e}")
    return results

# Time and run the process
start_time = time.time()
faithfulness_scores = asyncio.run(evaluate_all(df))
end_time = time.time()

print(f"Evaluation completed in {end_time - start_time:.2f} seconds.")

Evaluating:   0%|          | 0/1000 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 1000/1000 [04:28<00:00,  3.73it/s]

Evaluation completed in 268.35 seconds.


In [37]:
# Step 1: Create DataFrame from prediction results
pred_df = pd.DataFrame([
    {"index": idx, "faithfulness_score": score, "pred_score": 1 - score}
    for item in faithfulness_scores
    if isinstance(item, tuple) and not isinstance(item[1], str)
    for idx, score in [item]  # safely unpack 1-item tuple
])

# Step 2: Join with ground truth labels
merged_df = pd.merge(pred_df, hallucination_df[["index", "hallucinated", "actual_output"]], on="index")

# Step 3: Round scores to get predicted labels
merged_df["pred_label"] = merged_df["pred_score"].round().astype(int)
merged_df["true_label"] = merged_df["hallucinated"].astype(int)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Step 4: Calculate metrics
accuracy = accuracy_score(merged_df["true_label"], merged_df["pred_label"])
precision = precision_score(merged_df["true_label"], merged_df["pred_label"], zero_division=0)
recall = recall_score(merged_df["true_label"], merged_df["pred_label"], zero_division=0)
f1 = f1_score(merged_df["true_label"], merged_df["pred_label"], zero_division=0)

# Output
print(f"Accuracy: {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f}")

Accuracy: 0.690
Precision: 0.748
Recall: 0.503
F1 Score: 0.602
